# 01 — Baselines walkthrough: three ways to trade weekly vol

The kit ships **three reference strategies**. None of them is the answer you should
submit — they are the *benchmarks* that make the lecture concrete. We run all three
on the **same seeded price path** and watch their equity curves.

| Baseline | What it does | What it teaches |
|---|---|---|
| `naive_short_straddle` | Sell the ATM call **and** put, no hedge, no risk layer | **Short gamma blows up** — the bomb |
| `buy_and_hold_call` | Buy one ATM call and hold | **Theta bleed** — optionality isn't free |
| `delta_hedged_vol_seller` | Sell a (smaller) straddle **and re-hedge delta each tick** + kill-switch | **The value of hedging** — the benchmark to beat |

Tie this back to the lecture: a short option is **short gamma / short vega**; a long
option is **long gamma / long vega but pays theta**; hedging delta converts a
directional bet into a bet on **realised vol vs implied vol**.

In [1]:
import os, sys
# Notebooks run from the notebooks/ folder; hop up to the repo root so the
# `botkit`/`strategies` imports and the data/ + runs/ paths all resolve.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

working dir: /Users/kalam/Podcast/iba-course-derivatives/f405-options-bot


## Run helper

Each baseline exposes a `Strategy` and a `RiskManager`. We run them through the same
`botkit.runner` the CLI uses, then load the resulting `equity.csv`. Results are cached
so re-plotting is instant.

In [2]:
import importlib
import pandas as pd
from botkit.config import Config
from botkit import runner

# name -> (module, Strategy class, RiskManager class)
BASELINES = {
    "naive_short_straddle":     ("strategies.baselines.naive_short_straddle", "NaiveShortStraddle", "NaiveRisk"),
    "buy_and_hold_call":        ("strategies.baselines.buy_and_hold_call", "BuyAndHoldCall", "BuyHoldRisk"),
    "delta_hedged_vol_seller":  ("strategies.baselines.delta_hedged_vol_seller", "DeltaHedgedVolSeller", "HedgedRisk"),
}

_cache = {}
def run_baseline(name, seed):
    """Run one baseline in sim at a given seed; return (summary, equity DataFrame)."""
    key = (name, seed)
    if key in _cache:
        return _cache[key]
    mod, strat, risk = BASELINES[name]
    m = importlib.import_module(mod)
    cfg = Config.load("config.example.yaml")
    cfg.sim_seed = seed
    cfg.journal_dir = f"runs/wt_{name}_seed{seed}"
    summary = runner.run(cfg, getattr(m, strat)(), getattr(m, risk)())
    eq = pd.read_csv(f"{cfg.journal_dir}/equity.csv", parse_dates=["iso"])
    _cache[key] = (summary, eq)
    return _cache[key]

import plotly.graph_objects as go

COLORS = {"naive_short_straddle": "#d62728",
          "buy_and_hold_call": "#1f77b4",
          "delta_hedged_vol_seller": "#2ca02c"}

def equity_fig(seed, title):
    """Plot all three baselines' equity, normalised to 100% at the start."""
    fig = go.Figure()
    for name in BASELINES:
        _, eq = run_baseline(name, seed)
        base = eq["equity_usd"].iloc[0]
        fig.add_trace(go.Scatter(
            x=eq["iso"], y=100 * eq["equity_usd"] / base,
            name=name, line=dict(color=COLORS[name])))
    fig.add_hline(y=100, line_dash="dot", line_color="grey")
    fig.update_layout(title=title, xaxis_title="time",
                      yaxis_title="equity (% of start)",
                      template="plotly_white", height=430)
    return fig

## A calm week (seed 0)

First, a quiet path where the underlying drifts but never lurches. This is the
environment short-vol *loves*.

In [3]:
equity_fig(0, "Calm week (seed 0) — equity curves")

**What you see on a calm path:**
- **Naive short straddle (red)** posts the best *headline* return — it just keeps the
  premium as theta decays both legs. But notice how *jumpy* the line is: short gamma
  means every wiggle moves the mark hard. The return comes with hidden risk.
- **Buy & hold call (blue)** drifts **down**: with no big up-move, time value bleeds
  out of the long call every tick. That downward grind *is* theta.
- **Delta-hedged vol seller (green)** is nearly **flat and smooth**. It harvested the
  same theta but paid a little of it back hedging — buying calm at the cost of a
  better risk-adjusted ride. On a quiet week that looks unexciting. The next week is
  why it exists.

## A shock week (seed 7): the blow-up

Now a path with a real move into expiry — the week that decides who survives.

In [4]:
equity_fig(7, "Shock week (seed 7) — the blow-up")

**What you see when the market moves:**
- **Naive short straddle (red) gets liquidated.** As the underlying runs, the short
  straddle's delta runs *against* it, and short **gamma** makes the loss *accelerate*
  exactly when it hurts most — near expiry. With no risk layer, equity craters through
  zero: the runner's liquidation trip flattens it. **This is the bomb the assignment
  warns you not to ship.**
- **Delta-hedged vol seller (green) survives.** Re-hedging kept net delta near zero, so
  the move did far less damage; and its **kill-switch flattens at a ~20% drawdown**,
  capping the loss *before* liquidation. A bad week, not a fatal one.
- **Buy & hold call (blue)** loses too, but its loss is **bounded** by the premium it
  paid — the comforting flip-side of being long an option.

## Scoreboard across several seeds

One path can flatter anyone. Here is each baseline over five seeded weeks. The grader
cares about **risk-adjusted** return (`return / max(drawdown, 2%)`) and zeroes anything
that **blew up** (got liquidated, or ever fell below 25% of start).

In [5]:
def metrics(eq):
    e = eq["equity_usd"].to_numpy()
    ret = e[-1] / e[0] - 1.0
    peak, mdd = e[0], 0.0
    for x in e:
        peak = max(peak, x)
        mdd = max(mdd, (peak - x) / peak)
    blew = bool((eq["liquidated"] == 1).any()) or (e.min() < 0.25 * e[0])
    return ret, mdd, blew

recs = []
for seed in [0, 1, 2, 3, 7]:
    for name in BASELINES:
        s, eq = run_baseline(name, seed)
        ret, mdd, blew = metrics(eq)
        recs.append({"seed": seed, "baseline": name, "return": ret,
                     "max_dd": mdd, "risk_adj": ret / max(mdd, 0.02),
                     "blew_up": blew, "fills": s["fills"]})
board = pd.DataFrame(recs)
# Pivot risk-adjusted score into a seed x baseline grid for an at-a-glance read,
# then show the per-run detail (note the blew_up flags) underneath.
pivot = board.pivot(index="seed", columns="baseline", values="risk_adj").round(2)
print("risk-adjusted score (return / max(drawdown, 2%)) by seed:")
display(pivot)
board.style.format({"return": "{:+.1%}", "max_dd": "{:.1%}", "risk_adj": "{:.2f}"})

risk-adjusted score (return / max(drawdown, 2%)) by seed:


baseline,buy_and_hold_call,delta_hedged_vol_seller,naive_short_straddle
seed,,,
0,0.22,-0.02,0.85
1,-0.71,-0.62,-0.46
2,-0.49,-0.71,3.73
3,-0.42,-0.65,2.61
7,-0.99,-1.00,-1.00


,seed,baseline,return,max_dd,risk_adj,blew_up,fills
0,0,naive_short_straddle,+19.8%,23.2%,0.85,False,2
1,0,buy_and_hold_call,+1.8%,8.4%,0.22,False,1
2,0,delta_hedged_vol_seller,-0.1%,4.9%,-0.02,False,15
3,1,naive_short_straddle,-22.3%,48.4%,-0.46,False,2
4,1,buy_and_hold_call,-8.3%,11.6%,-0.71,False,1
5,1,delta_hedged_vol_seller,-6.5%,10.5%,-0.62,False,14
6,2,naive_short_straddle,+30.3%,8.1%,3.73,False,2
7,2,buy_and_hold_call,-3.8%,7.7%,-0.49,False,1
8,2,delta_hedged_vol_seller,-6.9%,9.8%,-0.71,False,24
9,3,naive_short_straddle,+35.4%,13.6%,2.61,False,2


Read it by **drawdown and blow-ups**, not headline return. The naive straddle can win a
calm week big, but it carries the worst drawdowns and a `blew_up = True` somewhere —
and a blow-up scores **0** no matter the other weeks. The delta-hedged seller never
blows up and keeps drawdowns small: a worse headline, a far better *risk-adjusted* score.
That trade-off is the entire point.

## Why each baseline wins or loses — the lecture in one place

**1. Naive short straddle — short gamma, short vega.**  
Selling the ATM call and put collects two premiums and is delta-neutral *for an
instant*. But it is **short gamma**: as spot moves, your delta moves the wrong way, and
your losses are **convex** — they grow faster than the premium you took in. Gamma
explodes near expiry, so one move in the final days is enough to liquidate an
un-hedged, over-sized book. It is also **short vega**: a vol spike marks it against you
immediately. Great carry, fat fatal tail.

**2. Buy & hold call — long gamma/vega, but you pay theta.**  
A long call can only lose its premium (nice) but **bleeds theta every day** (not nice).
With no strong up-move, time decay grinds the position down — you watch equity slope
lower on a quiet path. Optionality is insurance you *rent*; holding it passively and
hoping is a slow loss.

**3. Delta-hedged vol seller — the benchmark to beat.**  
Same premium harvest, but **re-hedged to ~0 net delta every tick** (gamma scalping).
Neutralising direction turns the trade into a clean bet: **realised vol < implied vol**.
You give back some theta in hedging costs, but you no longer die on a move. Add a
**smaller size** and a **kill-switch that flattens before liquidation**, and you get a
boring, survivable equity curve — exactly what a good risk-adjusted score rewards.

### So how do you *beat* the green line?
You don't beat it by selling more vol — that's just the red line waiting to happen.
Beat it by **shrinking the denominator** (drawdown): bound net vega and gross size in
`MyRisk.vet()`, **cap the tail** (e.g. buy a cheap wing to cut the worst move), pick
your spots instead of always being short, and make `should_halt()` a real kill-switch.
The grader divides return by drawdown — a smaller drawdown is worth more than a bigger
headline. **Do not ship the naive answer.**